# Part II — Task A: Weighted Syndrome Decoder Exploration

This notebook provides the interactive workspace to develop and verify the **Task A** models before migrating them to `train.py`.

### Task Summary:
- **Dataset:** `ml/ml_syndrome_decoder_example.parquet` in MinIO.
- **Features ($X$):** 16 binary syndrome bits (4 rounds $\times$ 4 checks).
- **Target ($y$):** `logical_error_label` (Boolean: $0$ or $1$).
- **Weighting:** `sample_weight` (physical observation counts).
- **Models to compare:**
  1. Weighted Majority / Prior Baseline
  2. Weighted Logistic Regression

## 1. Imports and Platform Connections
Import standard scientific libraries (`numpy`, `pandas`, `sklearn`) and the course connection/helper modules from `quantum_lake_student`.

In [1]:
import io 
import pyarrow.parquet as pq
import numpy as np 
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import read_lake_object
from quantum_lake_student.ml import syndrome_model_input



## 2. Load the Prepared ML Table from MinIO
Read `ml/ml_syndrome_decoder_example.parquet` from the MinIO object store using the course connection helper.

In [2]:
settings = Settings.from_environment()
raw_bytes = read_lake_object(settings,"ml/ml_syndrome_decoder_example.parquet")

table = pq.read_table(io.BytesIO(raw_bytes))
df = table.to_pandas()
df.head()

,example_id,experiment_id,physical_fault_rate,syndrome_bits,round_count,check_count,logical_error_label,sample_weight,data_split
0,ml-syn-000011de0d7314b10c8f07e9f3d24f8ec48da0e...,qec-exp-8fe975d642bc5073a7443b660f1bb0f3465c14...,0.01,b'\x00\x00\x00\x00\x00\x01\x00\x01\x00\x01\x00...,4,4,True,7,train
1,ml-syn-0002186a785a760d99f696e6cac86948b055473...,qec-exp-8fe975d642bc5073a7443b660f1bb0f3465c14...,0.01,b'\x00\x00\x01\x00\x01\x00\x01\x00\x00\x01\x01...,4,4,False,4,train
2,ml-syn-0002d64fe840d97a776a02eb158f685db7b000c...,qec-exp-8fe975d642bc5073a7443b660f1bb0f3465c14...,0.01,b'\x00\x00\x01\x01\x01\x00\x00\x00\x01\x00\x01...,4,4,False,1,train
3,ml-syn-00030196d678ee1e78f9fa0f1186bf561455381...,qec-exp-8fe975d642bc5073a7443b660f1bb0f3465c14...,0.01,b'\x01\x00\x01\x00\x01\x00\x00\x01\x01\x01\x01...,4,4,False,1,train
4,ml-syn-00041c7e4b162a8f28e8279d6484c55ad540b7a...,qec-exp-8fe975d642bc5073a7443b660f1bb0f3465c14...,0.01,b'\x01\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00...,4,4,True,2469,train


## 3. Data Inspection and Sanity Checks
Inspect the loaded dataframe columns, row counts (expected: 75,598 rows), and data types.

In [3]:
df.count()
df.dtypes


example_id              object
experiment_id           object
physical_fault_rate    float64
syndrome_bits           object
round_count              int32
check_count              int32
logical_error_label       bool
sample_weight            int64
data_split              object
dtype: object

## 4. Extract Features ($X$), Target ($y$), and Physical Weights ($w$)
Convert the 16 bytes of `syndrome_bits` into a 2D binary matrix of shape `(N, 16)`, extract `logical_error_label` as $y$, and extract `sample_weight` as weights.

In [4]:
X = np.array([syndrome_model_input(row) for row in df["syndrome_bits"]])
y = df["logical_error_label"]
w = df["sample_weight"]

print("Shape of X:", X.shape)
print("First row:", X[0])


Shape of X: (75598, 16)
First row: [0 0 0 0 0 1 0 1 0 1 0 1 0 0 1 1]


## 5. Course-Mandated Train / Validation / Test Split
Partition the data strictly by `physical_fault_rate` using the course split rule:
- Validation: `pfr == 0.0005`
- Test: `pfr == 0.005`
- Train: The remaining 5 fault-rate files.

In [5]:
train_mask = (df["data_split"] == "train").to_numpy()
val_mask   = (df["data_split"] == "validation").to_numpy()
test_mask  = (df["data_split"] == "test").to_numpy()

# 1. Training Set
X_train = X[train_mask]
y_train = y[train_mask]
w_train = w[train_mask]

# 2. Validation Set
X_val = X[val_mask]
y_val = y[val_mask]
w_val = w[val_mask]

# 3. Test Set
X_test = X[test_mask]
y_test = y[test_mask]
w_test = w[test_mask]

## 6. Model 1: Weighted Majority / Prior Baseline
Fit a dummy prior classifier using `sample_weight` to establish the baseline performance bar.

In [6]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="prior")
baseline.fit(X_train, y_train, sample_weight=w_train)

from quantum_lake_student.ml import weighted_logical_error_rate

# 1. Get predictions on the test set
test_preds = baseline.predict(X_test)

# 2. Calculate the weighted Logical Error Rate (LER)
base_ler = weighted_logical_error_rate(y_test, test_preds, w_test)

print(f"Baseline Test LER: {base_ler:.6f}")
print("Unique predictions:", set(test_preds))

Baseline Test LER: 0.103950
Unique predictions: {np.False_}


## 7. Model 2: Train Weighted Logistic Regression
Train a `LogisticRegression` model on the training partition with `sample_weight` and a fixed random seed (`random_state=42`). Measure the training execution time.

In [7]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=42).fit(X_train,y_train,sample_weight=w_train)

# Check the default threshold on validation: the test split is used once, in Step 9
val_preds_default = clf.predict(X_val)

# 2. Calculate the weighted Logical Error Rate (LER) without overwriting base_ler
lr_val_ler = weighted_logical_error_rate(y_val, val_preds_default, w_val)

print(f"LogReg Validation LER (threshold 0.50): {lr_val_ler:.6f}")
print("Unique predictions:", set(val_preds_default))

LogReg Validation LER (threshold 0.50): 0.011511
Unique predictions: {np.False_, np.True_}


## 8. Threshold Tuning on Validation Data Only
Evaluate candidate decision thresholds (between 0.05 and 0.95) on the Validation set to find the threshold that minimizes Weighted Logical Error Rate (LER).

In [8]:
val_probs = clf.predict_proba(X_val)[:, 1]

best_threshold = 0.5
best_val_ler = float("inf")

for threshold in np.arange(0.05, 0.95, 0.05):
    # Predict True if probability >= threshold, else False
    val_preds = val_probs >= threshold
    
    # Calculate the weighted error rate on validation data
    val_ler = weighted_logical_error_rate(y_val, val_preds, w_val)
    print(f"Threshold: {threshold:.2f} --> Validation LER: {val_ler:.6f}")
    
    if val_ler < best_val_ler:
        best_val_ler = val_ler
        best_threshold = threshold
print("-" * 40)
print(f"Optimal Threshold on Validation: {best_threshold:.2f} (LER: {best_val_ler:.6f})")


Threshold: 0.05 --> Validation LER: 0.029467
Threshold: 0.10 --> Validation LER: 0.022842
Threshold: 0.15 --> Validation LER: 0.022363
Threshold: 0.20 --> Validation LER: 0.016611
Threshold: 0.25 --> Validation LER: 0.013974
Threshold: 0.30 --> Validation LER: 0.013656
Threshold: 0.35 --> Validation LER: 0.011581
Threshold: 0.40 --> Validation LER: 0.011549
Threshold: 0.45 --> Validation LER: 0.011523
Threshold: 0.50 --> Validation LER: 0.011511
Threshold: 0.55 --> Validation LER: 0.011494
Threshold: 0.60 --> Validation LER: 0.011487
Threshold: 0.65 --> Validation LER: 0.011481
Threshold: 0.70 --> Validation LER: 0.011482
Threshold: 0.75 --> Validation LER: 0.011467
Threshold: 0.80 --> Validation LER: 0.011456
Threshold: 0.85 --> Validation LER: 0.011454
Threshold: 0.90 --> Validation LER: 0.011441
----------------------------------------
Optimal Threshold on Validation: 0.90 (LER: 0.011441)


## 9. Final Evaluation on Held-Out Test Data
Evaluate both the Baseline and the Logistic Regression model on the Test set once using the optimal threshold.
Calculate:
- Weighted Logical-Error Rate (LER)
- Weighted Balanced Accuracy
- Weighted Brier Score
- Prediction Time

In [9]:
import time
from sklearn.metrics import balanced_accuracy_score, brier_score_loss


# 1. Predict probabilities on Test and time the execution
start_pred = time.perf_counter()
test_probs = clf.predict_proba(X_test)[:, 1]
pred_time = time.perf_counter() - start_pred

# 2. Apply your optimal threshold from Step 8 (0.90)
test_preds = test_probs >= best_threshold

# 3. Calculate all required metrics
test_ler = weighted_logical_error_rate(y_test, test_preds, w_test)
test_balanced_acc = balanced_accuracy_score(y_test, test_preds, sample_weight=w_test)
test_brier = brier_score_loss(y_test, test_probs, sample_weight=w_test)

print("=" * 45)
print("TASK A — FINAL TEST RESULTS (Logistic Regression)")
print("=" * 45)
print(f"Optimal Threshold used:   {best_threshold:.2f}")
print(f"Weighted Test LER:        {test_ler:.6f}  (Baseline was: 0.103950)")
print(f"Weighted Balanced Acc:    {test_balanced_acc:.4f}")
print(f"Weighted Brier Score:     {test_brier:.6f}")
print(f"Prediction Time:          {pred_time:.4f} seconds")

TASK A — FINAL TEST RESULTS (Logistic Regression)
Optimal Threshold used:   0.90
Weighted Test LER:        0.104156  (Baseline was: 0.103950)
Weighted Balanced Acc:    0.5015
Weighted Brier Score:     0.090956
Prediction Time:          0.0053 seconds


## 10. Inspect Learned Sensor Coefficients (Danger Weights)
Inspect `model.coef_` across the 16 sensor positions to see which sensors and time rounds carry the highest risk.

In [10]:
import pandas as pd
import numpy as np

# 1. Base risk (bias)
print(f"Base Intercept: {clf.intercept_[0]:.4f}")

# 2. Reshape the 16 weights into a 4x4 grid (4 rounds x 4 sensors)
coef_matrix = clf.coef_[0].reshape(4, 4)

df_coef = pd.DataFrame(
    coef_matrix,
    index=[f"Round {r+1}" for r in range(4)],
    columns=[f"Check {c}" for c in range(4)]
)

print("\nLearned Danger Weights (w_i):")
print(df_coef.round(4))

Base Intercept: -4.0325

Learned Danger Weights (w_i):
         Check 0  Check 1  Check 2  Check 3
Round 1   1.7339  -0.1644   2.2877  -0.0149
Round 2   0.7380   0.3823   1.0612   0.4053
Round 3   0.8880   0.2819   1.2820   0.2879
Round 4   1.5598   0.1169   1.5030   0.1772


## 11. Format Predictions and Metrics for Export
Format test predictions to match the schema of `results/part2/predictions.parquet` and assemble the metrics dictionary for `results/part2/metrics.json`.

In [11]:
import time
import json
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa

# 1. Measure training time if train_time isn't already defined
t0 = time.perf_counter()
clf.fit(X_train, y_train, sample_weight=w_train)
train_time = time.perf_counter() - t0

# 2. Measure prediction time
t0 = time.perf_counter()
test_probs = clf.predict_proba(X_test)[:, 1]
pred_time = time.perf_counter() - t0
test_preds = test_probs >= best_threshold

# 3. Build predictions DataFrame (using df["example_id"] directly)
df_preds = pd.DataFrame({
    "example_id": df.loc[test_mask, "example_id"].values,
    "model_id": "weighted_logistic_regression",
    "label": y_test.astype(bool),
    "prediction": test_preds.astype(bool),
    "probability": test_probs.astype(float),
    "split": "test",
})

# 4. Build metrics dictionary
task_a_metrics = {
    "task_a_syndrome_decoder": {
        "baseline_prior": {
            "weighted_logical_error_rate": float(base_ler),
            "brier_score": None,
        },
        "weighted_logistic_regression": {
            "optimal_threshold": float(best_threshold),
            "weighted_logical_error_rate": float(test_ler),
            "weighted_balanced_accuracy": float(test_balanced_acc),
            "weighted_brier_score": float(test_brier),
            "training_time_seconds": float(train_time),
            "prediction_time_seconds": float(pred_time),
        }
    }
}

# 5. Save files to results/part2/
results_dir = Path("results/part2")
results_dir.mkdir(parents=True, exist_ok=True)

df_preds.to_parquet(results_dir / "predictions.parquet", index=False)
with open(results_dir / "metrics.json", "w") as f:
    json.dump(task_a_metrics, f, indent=2)

print("Saved predictions.parquet and metrics.json successfully to results/part2/!")
df_preds.head()

Saved predictions.parquet and metrics.json successfully to results/part2/!


,example_id,model_id,label,prediction,probability,split
5,ml-syn-00049a112d18cc831f6376425b4c36a24422380...,weighted_logistic_regression,False,False,0.639153,test
7,ml-syn-0004e43ef72f5fb8e147152f8c74ebc4e63d7cd...,weighted_logistic_regression,False,False,0.059402,test
8,ml-syn-0007876d36ce9675bad13c34022d6bb9e6aead0...,weighted_logistic_regression,False,False,0.632680,test
22,ml-syn-0011eaba3075d8eb306bb9160520244f41a2ec6...,weighted_logistic_regression,False,False,0.523484,test
31,ml-syn-001cdeb84798ad9aaa09ccdbeb611603dbc82eb...,weighted_logistic_regression,True,False,0.650834,test


# Task A: In-Depth Notes & Spatio-Temporal Insight

*Numbers below are from the training split.*

## 1. The 16 bits are a 4 × 4 grid

They are 4 checks measured over 4 rounds, stored round by round. The same check in every round shares the same `index % 4` (check 0 is indices 0, 4, 8, 12).

```text
           Check 0   Check 1   Check 2   Check 3
Round 1       0         1         2         3
Round 2       4         5         6         7
Round 3       8         9        10        11
Round 4      12        13        14        15
```

## 2. What a `1` means

A `1` means the check **changed** since the previous round. It does not mean the check is "on".

* **A single `1`** is a real error on a data qubit. On check 0 or 2 it flips the label about 97% of the time.
* **Two `1`s in a row on the same check** are one wrong readout. The label stays 0.

So the label depends on whether a check fires an **odd or even** number of times, not on how long it stays active:

| Check 0 fires in rounds | Shots with label 1 |
| --- | --- |
| 1 | 97% |
| 1, 2 | 0.1% |
| 1, 2, 3 | 96% |
| 1, 2, 3, 4 | 0.4% |

A label of 1 means the logical value flipped. That is what the decoder must predict; it does not mean error correction failed.

## 3. Why logistic regression fails

The model adds up one weight per bit. A sum cannot express "one event means error, two events mean no error":

| Round 1 | Round 2 | Shots with label 1 |
| --- | --- | --- |
| no event | no event | 0% |
| event | no event | 97% |
| no event | event | 97% |
| event | event | 0.1% |

If each event alone pushes the score over the threshold, both together push it even higher, never back under. This "one or the other, but not both" pattern is called an exclusive-or (XOR).

**Result:** test LER 0.104156, against 0.103950 for always predicting "no error". The model is no better than the baseline.

## 4. What the coefficients show

```text
Intercept: -4.03
           Check 0   Check 1   Check 2   Check 3
Round 1      1.73     -0.16      2.29     -0.01
Round 2      0.74      0.38      1.06      0.41
Round 3      0.89      0.28      1.28      0.29
Round 4      1.56      0.12      1.50      0.18
```

* **Checks 0 and 2 get large weights; checks 1 and 3 get almost none.** A single event on check 0 or 2 flips the label 97% of the time, on check 1 or 3 only 3%. This comes from where the checks sit in the code, not from different check types: the dataset's source paper uses only one type of check.
* **Rounds 1 and 4 weigh more than rounds 2 and 3.** An event in a middle round is more often half of a wrong-readout pair (which does not flip the label), because it has a neighboring round on both sides.
* **The intercept is not a real failure rate.** It gives a syndrome with no events a 1.7% chance of failure, but in the data such shots fail only 0.0024% of the time.

## Takeaway

The label depends on odd-or-even counts of events, which a weighted sum cannot represent. So logistic regression lands at the baseline, and that is the expected, honest result.